In [ ]:
# Cell 1: Import Libraries
import math
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import logging
import os
import pickle
from pathlib import Path
import copy

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.distributed as dist
from torch.utils.data import Dataset, DataLoader
from torch.nn.parallel import DistributedDataParallel as DDP

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")

print("All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

All libraries imported successfully!
PyTorch version: 2.9.0+cu126
CUDA available: True


In [ ]:
# Cell 2: Configuration
import random
import numpy as np

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

config = {
    'data_path': '/content/AMAZON_BEAUTY_27K_USERS.csv',
    'save_dir': './checkpoints',
    'log_dir': './logs',
    'max_len': 40,
    'hidden_units': 256,
    'num_heads': 2,
    'num_layers': 2,
    'dropout_rate': 0.1,
    'lr': 1e-4,
    'batch_size': 256,
    'num_epochs': 50,
    'num_workers': 2,
    'mask_prob': 0.4,
    'weight_decay': 0.0,
    'patience': 6,
    'evaluate_every': 2,
    'gradient_clip': 1.0,
    'train_negative_sample_size': 0,
    'test_negative_sample_size': 100,
    'train_negative_seed': 0,
    'test_negative_seed': 98765,
    'metric_ks': [5, 10, 15],
    'best_metric': 'HitRate@10',
    'enable_lr_schedule': False,
    'decay_step': 25,
    'gamma': 0.5,
    'random_seed': 2025,
}

set_seed(config['random_seed'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
#  Logging Setup Function
def setup_logging(log_dir, log_name='train'):
    if not os.path.exists(log_dir):
        os.makedirs(log_dir)

    log_path = os.path.join(log_dir, f'{log_name}.log')

    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(log_path),
            logging.StreamHandler()
        ]
    )

    logging.info(f"Logging to {log_path}")
    return logging.getLogger(__name__)

In [ ]:
class DataProcessor:
    def __init__(self, df):
        self.df = df.copy()
        self._process_data()

    def _process_data(self):
        self.item_encoder, self.item_decoder = self._generate_encoder_decoder(self.df['ProductId'])
        self.user_encoder, self.user_decoder = self._generate_encoder_decoder(self.df['UserId'])

        self.num_item = len(self.item_encoder)
        self.num_user = len(self.user_encoder)

        self.df['ProductType'] = self.df['ProductType'].astype(str)
        self.product_types = sorted(self.df['ProductType'].unique())
        self.num_types = len(self.product_types)

        self._item_to_type = {}
        for _, row in self.df[['ProductId', 'ProductType']].drop_duplicates().iterrows():
            item_id = self.item_encoder[row['ProductId']] + 1
            type_idx = self.product_types.index(row['ProductType']) + 1
            self._item_to_type[item_id] = type_idx

        self.df['item_idx'] = self.df['ProductId'].apply(lambda x: self.item_encoder[x] + 1)
        self.df['user_idx'] = self.df['UserId'].apply(lambda x: self.user_encoder[x])
        self.df = self.df.sort_values(['user_idx', 'Timestamp'])

        self.user_train, self.user_valid, self.user_test, self.types_seq = self._generate_sequences()

    def _generate_encoder_decoder(self, col):
        encoder = {val: idx for idx, val in enumerate(col.unique())}
        decoder = {idx: val for val, idx in encoder.items()}
        return encoder, decoder


    def _generate_sequences(self):
        user_train, user_valid, user_test, types_seq = {}, {}, {}, {}
        grouped = self.df.groupby('user_idx')['item_idx'].apply(list)

        for user, seq in grouped.items():
            if len(seq) < 5:
                continue

            user_train[user] = seq[:-2]
            user_valid[user] = seq[-2]
            user_test[user] = seq[-1]

            types_seq[user] = [self.get_item_type(item) for item in seq[:-2]]

        return user_train, user_valid, user_test, types_seq

    def get_item_type(self, item_idx):

        if item_idx == 0:
            return 0
        return self._item_to_type.get(item_idx, 0)

In [ ]:
#  NegativeSampler Class
class NegativeSampler:
    def __init__(self, user_train, user_valid, user_test, num_items, sample_size, seed, save_path):
        self.user_train = user_train
        self.user_valid = user_valid
        self.user_test = user_test
        self.num_items = num_items
        self.sample_size = sample_size
        self.seed = seed
        self.save_path = save_path
        self._all_items = set(range(1, num_items + 1))

    def generate_negative_samples(self):
        np.random.seed(self.seed)
        negative_samples = {}

        logging.info(f"Generating negative samples with seed={self.seed}")

        for user in tqdm(self.user_train.keys(), desc="Negative Sampling"):
            seen = set(self.user_train[user])
            if self.user_valid and user in self.user_valid:
                seen.add(self.user_valid[user])
            if self.user_test and user in self.user_test:
                seen.add(self.user_test[user])

            candidates = list(self._all_items - seen)

            if len(candidates) >= self.sample_size:
                samples = np.random.choice(candidates, self.sample_size, replace=False).tolist()
            else:
                samples = candidates

            negative_samples[user] = samples

        return negative_samples

    def get_negative_samples(self):
        if os.path.exists(self.save_path):
            logging.info(f"Loading negative samples from {self.save_path}")
            with open(self.save_path, 'rb') as f:
                negative_samples = pickle.load(f)
        else:
            logging.info(f"Generating new negative samples")
            negative_samples = self.generate_negative_samples()

            os.makedirs(os.path.dirname(self.save_path), exist_ok=True)
            with open(self.save_path, 'wb') as f:
                pickle.dump(negative_samples, f)
            logging.info(f"Saved negative samples to {self.save_path}")

        return negative_samples

In [ ]:
class BERTTrainDataset(Dataset):
    def __init__(self, data_processor, max_len, mask_prob, rng):
        self.data_processor = data_processor
        self.user_train = data_processor.user_train
        self.users = sorted([u for u in self.user_train if len(self.user_train[u]) > 0])

        self.max_len = max_len
        self.mask_prob = mask_prob
        self.rng = rng

        self.num_items = data_processor.num_item
        self.mask_token = self.num_items + 1

        self.num_types = data_processor.num_types
        self.type_mask_token = self.num_types + 1

        self._all_items = set(range(1, self.num_items + 1))

    def __len__(self):
        return len(self.users)

    def __getitem__(self, index):
        user = self.users[index]
        seq = self.user_train[user][-self.max_len:]

        tokens, types_seq, labels = [], [], []

        for s in seq:
            t = self.data_processor.get_item_type(s)
            prob = self.rng.random()

            if prob < self.mask_prob:
                prob /= self.mask_prob

                # 80% -> MASK item,mask type
                if prob < 0.8:
                    tokens.append(self.mask_token)
                    types_seq.append(self.type_mask_token)

                # 10% -> random item + type tương ứng
                elif prob < 0.9:
                    rnd_item = self.rng.randint(1, self.num_items)
                    tokens.append(rnd_item)
                    types_seq.append(
                        self.data_processor.get_item_type(rnd_item)
                    )

                # 10% -> keep original
                else:
                    tokens.append(s)
                    types_seq.append(t)

                labels.append(s)

            else:
                tokens.append(s)
                types_seq.append(t)
                labels.append(0)

        pad_len = self.max_len - len(tokens)

        tokens = [0] * pad_len + tokens
        labels = [0] * pad_len + labels
        types_seq = [0] * pad_len + types_seq

        return (
            torch.LongTensor(tokens),
            torch.LongTensor(types_seq),
            torch.LongTensor(labels)
        )


In [ ]:
class BERTEvalDataset(Dataset):
    def __init__(self, data_processor, user_target, max_len, negative_samples):
        self.data_processor = data_processor
        self.user_train = data_processor.user_train
        self.user_target = user_target
        self.users = sorted(self.user_train.keys())

        self.max_len = max_len
        self.mask_token = data_processor.num_item + 1
        self.negative_samples = negative_samples

    def __len__(self):
        return len(self.users)

    def __getitem__(self, index):
        user = self.users[index]
        seq = self.user_train[user]
        answer = self.user_target[user]

        seq = seq + [self.mask_token]
        seq = seq[-self.max_len:]

        types = [self.data_processor.get_item_type(item) for item in seq]

        padding_len = self.max_len - len(seq)
        seq = [0] * padding_len + seq
        types = [0] * padding_len + types

        negatives = self.negative_samples[user]
        candidates = [answer] + negatives
        labels = [1] + [0] * len(negatives)

        return (
            torch.LongTensor(seq),
            torch.LongTensor(types),
            torch.LongTensor(candidates),
            torch.LongTensor(labels)
        )

In [ ]:
#  Embedding Layer Classes
class PositionalEmbedding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()
        self.pe = nn.Embedding(max_len, d_model)

    def forward(self, x):
        batch_size = x.size(0)
        return self.pe.weight.unsqueeze(0).repeat(batch_size, 1, 1)


class TokenEmbedding(nn.Embedding):
    def __init__(self, vocab_size, embed_size):
        super().__init__(vocab_size, embed_size, padding_idx=0)


class TypeEmbedding(nn.Module):
    def __init__(self, num_types, embed_size):
        super().__init__()
        self.embedding = nn.Embedding(num_types + 2, embed_size, padding_idx=0)

    def forward(self, type_indices):
        return self.embedding(type_indices)


class BERTEmbedding(nn.Module):
    def __init__(self, vocab_size, num_types, embed_size, max_len, dropout=0.1):
        super().__init__()
        self.token = TokenEmbedding(vocab_size, embed_size)
        self.position = PositionalEmbedding(max_len, embed_size)
        self.type_emb = TypeEmbedding(num_types, embed_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, sequence, type_indices):
        x = self.token(sequence) + self.position(sequence) + self.type_emb(type_indices)
        return self.dropout(x)

**MODEL ARCHITECTURE**


In [ ]:
class Attention(nn.Module):
    """Tính toán toán học của Dot-Product Attention"""
    def forward(self, query, key, value, mask=None, dropout=None):
        scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(query.size(-1))

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        p_attn = F.softmax(scores, dim=-1)

        if dropout is not None:
            p_attn = dropout(p_attn)

        return torch.matmul(p_attn, value), p_attn

In [ ]:
#  Multi-Head Attention and Feed Forward Network
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0

        self.d_k = d_model // num_heads
        self.num_heads = num_heads

        # Sử dụng nn.ModuleList để chứa 3 lớp Linear cho Q, K, V
        self.linear_layers = nn.ModuleList([
            nn.Linear(d_model, d_model) for _ in range(3)
        ])

        # Output projection
        self.output_linear = nn.Linear(d_model, d_model)

        self.attention = Attention()
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        # Thực hiện chiếu tuyến tính (linear projections) cho cả Q, K, V cùng lúc
        # Kết quả trả về là list gồm [Q, K, V] đã được tách Head và Transpose
        query, key, value = [
            layer(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
            for layer, x in zip(self.linear_layers, (query, key, value))
        ]

        # Áp dụng Scaled Dot-Product Attention
        x, attn = self.attention(query, key, value, mask=mask, dropout=self.dropout)

        # Ghép các head lại (Concat) bằng cách dùng view
        # Chuyển từ (batch, num_heads, seq_len, d_k) về (batch, seq_len, d_model)
        x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads * self.d_k)

        # Áp dụng lớp Linear cuối cùng cho đầu ra
        return self.output_linear(x)


class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(math.sqrt(2 / math.pi) * (x + 0.044715 * torch.pow(x, 3))))


class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = GELU()

    def forward(self, x):
        return self.w_2(self.dropout(self.activation(self.w_1(x))))


class LayerNorm(nn.Module):
    def __init__(self, features, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(features))
        self.beta = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.gamma * (x - mean) / (std + self.eps) + self.beta


class SublayerConnection(nn.Module):
    """Bao bọc Residual Connection và Layer Normalization theo kiểu Pre-Norm"""
    def __init__(self, size, dropout):
        super().__init__()
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer_fn):
        # Pre-Norm: Chuẩn hóa x trước khi đưa vào lớp con (sublayer_fn)
        return x + self.dropout(sublayer_fn(self.norm(x)))

In [ ]:
# Transformer Block
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)

        self.input_sublayer = SublayerConnection(d_model, dropout)
        self.output_sublayer = SublayerConnection(d_model, dropout)

        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x, mask):
        # qua lớp Attention
        x = self.input_sublayer(x, lambda _x: self.attention(_x, _x, _x, mask=mask))

        # qua lớp Feed Forward
        x = self.output_sublayer(x, self.feed_forward)

        return self.dropout(x)

In [ ]:
# BERT4Rec Model
class BERT4Rec(nn.Module):
    def __init__(self, num_items, num_types, d_model, num_layers, num_heads, max_len, dropout):
        super().__init__()
        self.num_items = num_items
        self.mask_token = num_items + 1

        self.embedding = BERTEmbedding(
            vocab_size=num_items + 2,
            num_types=num_types,
            embed_size=d_model,
            max_len=max_len,
            dropout=dropout
        )

        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_model * 4, dropout)
            for _ in range(num_layers)
        ])

        self.out = nn.Linear(d_model, num_items + 1)

    def forward(self, seq, types):
        mask = (seq > 0).unsqueeze(1).repeat(1, seq.size(1), 1).unsqueeze(1)

        x = self.embedding(seq, types)

        for transformer in self.transformer_blocks:
            x = transformer(x, mask)

        return self.out(x)

In [ ]:
def calculate_metrics(scores, labels, ks):
    metrics = {}

    rank = (-scores).argsort(dim=1)
    for k in ks:
        cut = rank[:, :k]
        hits = labels.gather(1, cut).float()

        hit_rate = (hits.sum(1) > 0).float().mean()
        metrics[f'HitRate@{k}'] = hit_rate.item()

        position = torch.arange(2, 2 + k, device=hits.device).float()
        weights = 1 / torch.log2(position)
        dcg = (hits * weights).sum(1)

        answer_count = labels.sum(1)
        idcg = torch.tensor([weights[:min(int(n), k)].sum() for n in answer_count], device=dcg.device)

        ndcg = (dcg / idcg).mean()
        metrics[f'NDCG@{k}'] = ndcg.item()

    return metrics

In [ ]:
class Trainer:
    def __init__(self, model, data_processor, config, device, is_distributed=False):
        self.model = model
        self.data_processor = data_processor
        self.config = config
        self.device = device
        self.is_distributed = is_distributed

        self.criterion = nn.CrossEntropyLoss(ignore_index=0)
        self.optimizer = self._create_optimizer()

        if config['enable_lr_schedule']:
            self.lr_scheduler = optim.lr_scheduler.StepLR(
                self.optimizer,
                step_size=config['decay_step'],
                gamma=config['gamma']
            )

        self.best_metric = -1
        self.best_state = None
        self.patience_counter = 0

        self.train_losses = []
        self.val_metrics = []

        self._prepare_dataloaders()

    def _create_optimizer(self):
        return optim.Adam(
            self.model.parameters(),
            lr=self.config['lr'],
            weight_decay=self.config['weight_decay']
        )

    def _prepare_dataloaders(self):
        # TRAIN DATALOADER
        rng = random.Random(self.config['random_seed'])

        train_dataset = BERTTrainDataset(
            self.data_processor,
            self.config['max_len'],
            self.config['mask_prob'],
            rng
        )

        # NEGATIVE SAMPLING FOR EVALUATION
        train_neg_path = os.path.join(
            self.config['save_dir'],
            f'negative_train_size{self.config["train_negative_sample_size"]}_seed{self.config["train_negative_seed"]}.pkl'
        )
        train_neg_sampler = NegativeSampler(
            self.data_processor.user_train,
            self.data_processor.user_valid,
            self.data_processor.user_test,
            self.data_processor.num_item,
            self.config['train_negative_sample_size'],
            self.config['train_negative_seed'],
            train_neg_path
        )

        test_neg_path = os.path.join(
            self.config['save_dir'],
            f'negative_test_size{self.config["test_negative_sample_size"]}_seed{self.config["test_negative_seed"]}.pkl'
        )
        test_neg_sampler = NegativeSampler(
            self.data_processor.user_train,
            self.data_processor.user_valid,
            self.data_processor.user_test,
            self.data_processor.num_item,
            self.config['test_negative_sample_size'],
            self.config['test_negative_seed'],
            test_neg_path
        )

        test_negatives = test_neg_sampler.get_negative_samples()

        # VALIDATION DATALOADER
        val_dataset = BERTEvalDataset(
            self.data_processor,
            self.data_processor.user_valid,
            self.config['max_len'],
            test_negatives
        )

        # TEST DATALOADER
        test_dataset = BERTEvalDataset(
            self.data_processor,
            self.data_processor.user_test,
            self.config['max_len'],
            test_negatives
        )


        if self.is_distributed:
            train_sampler = torch.utils.data.distributed.DistributedSampler(
                train_dataset,
                shuffle=True,
                seed=self.config['random_seed']
            )
        else:
            train_sampler = None

        # Chỉ giữ lại DataLoader cho train, val và test
        self.train_loader = DataLoader(
            train_dataset,
            batch_size=self.config['batch_size'],
            sampler=train_sampler,
            shuffle=(train_sampler is None),
            num_workers=self.config['num_workers'],
            pin_memory=True,
            worker_init_fn=self._worker_init_fn
        )

        self.val_loader = DataLoader(
            val_dataset,
            batch_size=self.config['batch_size'],
            shuffle=False,
            num_workers=self.config['num_workers'],
            pin_memory=True
        )

        self.test_loader = DataLoader(
            test_dataset,
            batch_size=self.config['batch_size'],
            shuffle=False,
            num_workers=self.config['num_workers'],
            pin_memory=True
        )


    def _worker_init_fn(self, worker_id):
        """SỬA: Set seed cho mỗi worker để reproducibility"""
        worker_seed = self.config['random_seed'] + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)

In [ ]:
# Trainer Class - Part 2 (Training Methods)

def train_epoch(self):
    self.model.train()
    total_loss = 0

    for batch in tqdm(self.train_loader, desc='Training', leave=False):
        seqs, types, labels = [x.to(self.device) for x in batch]

        self.optimizer.zero_grad()

        logits = self.model(seqs, types)
        logits = logits.view(-1, logits.size(-1))
        labels = labels.view(-1)

        loss = self.criterion(logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=self.config['gradient_clip'])

        self.optimizer.step()

        total_loss += loss.item()

    return total_loss / len(self.train_loader)

def validate(self):
    self.model.eval()

    all_scores = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(self.val_loader, desc='Validating', leave=False):
            seqs, types, candidates, labels = [x.to(self.device) for x in batch]

            logits = self.model(seqs, types)
            scores = logits[:, -1, :]
            scores = scores.gather(1, candidates)

            all_scores.append(scores)
            all_labels.append(labels)

    all_scores = torch.cat(all_scores, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    metrics = calculate_metrics(all_scores, all_labels, self.config['metric_ks'])

    return metrics

Trainer.train_epoch = train_epoch
Trainer.validate = validate

In [ ]:
# Trainer Class - Part 3 (Main Training Loop)

def train(self):
    logging.info("Starting training...")

    for epoch in range(self.config['num_epochs']):
        if self.config['enable_lr_schedule']:
            self.lr_scheduler.step()

        train_loss = self.train_epoch()
        self.train_losses.append(train_loss)

        logging.info(f"Epoch {epoch+1}/{self.config['num_epochs']}")
        logging.info(f"  Train Loss: {train_loss:.4f}")

        if (epoch + 1) % self.config['evaluate_every'] == 0 or epoch == self.config['num_epochs'] - 1:
            val_metrics = self.validate()
            self.val_metrics.append(val_metrics)

            logging.info(f"  Validation Metrics:")
            for metric, value in val_metrics.items():
                logging.info(f"    {metric}: {value:.4f}")

            current_metric = val_metrics[self.config['best_metric']]

            if current_metric > self.best_metric:
                self.best_metric = current_metric
                self.best_state = copy.deepcopy(self.model.state_dict())
                self.patience_counter = 0

                self._save_checkpoint(epoch)
            else:
                self.patience_counter += 1
                if self.patience_counter >= self.config['patience']:
                    logging.info(f"Early stopping at epoch {epoch+1}")
                    break

    if self.best_state is not None:
        self.model.load_state_dict(self.best_state)

Trainer.train = train

In [ ]:
# Trainer Class - Part 4 (Testing)

def test(self):
    logging.info("Testing best model...")

    self.model.eval()
    all_scores = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(self.test_loader, desc='Testing',leave=False):
            seqs, types, candidates, labels = [x.to(self.device) for x in batch]

            logits = self.model(seqs, types)
            scores = logits[:, -1, :]
            scores = scores.gather(1, candidates)

            all_scores.append(scores)
            all_labels.append(labels)

    all_scores = torch.cat(all_scores, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    test_metrics = calculate_metrics(all_scores, all_labels, self.config['metric_ks'])

    logging.info("Test Results:")
    for metric, value in test_metrics.items():
        logging.info(f"  {metric}: {value:.4f}")

    return test_metrics

Trainer.test = test

In [ ]:
#  Trainer Class - Part 5 (Utility Methods)
def _save_checkpoint(self, epoch):
    if not os.path.exists(self.config['save_dir']):
        os.makedirs(self.config['save_dir'])

    checkpoint_path = os.path.join(self.config['save_dir'], 'Bert4Rec.pth')

    torch.save({
        'epoch': epoch,
        'model_state_dict': self.model.state_dict(),
        'optimizer_state_dict': self.optimizer.state_dict(),
        'best_metric': self.best_metric,
    }, checkpoint_path)

    logging.info(f"Checkpoint saved to {checkpoint_path}")

def load_checkpoint(self, checkpoint_path):
    logging.info(f"Loading checkpoint from {checkpoint_path}")

    checkpoint = torch.load(checkpoint_path, map_location=self.device)
    self.model.load_state_dict(checkpoint['model_state_dict'])
    self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    self.best_metric = checkpoint['best_metric']

    logging.info(f"Loaded model from epoch {checkpoint['epoch']}")
    logging.info(f"Best metric: {self.best_metric:.4f}")

Trainer._save_checkpoint = _save_checkpoint
Trainer.load_checkpoint = load_checkpoint

In [ ]:
def main_single_gpu():
    set_seed(config['random_seed'])

    logger = setup_logging(config['log_dir'], 'train_single')
    logger.info("Starting single GPU training")

    df = pd.read_csv(config['data_path'])

    data_processor = DataProcessor(df)

    # Khởi tạo mô hình BERT4Rec
    model = BERT4Rec(
        num_items=data_processor.num_item,
        num_types=data_processor.num_types,
        d_model=config['hidden_units'],
        num_layers=config['num_layers'],
        num_heads=config['num_heads'],
        max_len=config['max_len'],
        dropout=config['dropout_rate']
    ).to(device)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    trainer = Trainer(model, data_processor, config, device, is_distributed=False)

    trainer.train()

    test_metrics = trainer.test()
    logger.info(f"Test Results: {test_metrics}")
    return trainer, test_metrics

In [ ]:
# Run Training
trainer, test_metrics = main_single_gpu()
print("Training completed!")
print("\nTest Metrics:")
for metric, value in test_metrics.items():
    print(f"  {metric}: {value:.4f}")

2026-01-07 18:32:52,929 - root - INFO - Logging to ./logs/train_single.log
2026-01-07 18:32:52,932 - __main__ - INFO - Starting single GPU training
2026-01-07 18:33:02,752 - root - INFO - Generating new negative samples
2026-01-07 18:33:02,753 - root - INFO - Generating negative samples with seed=98765
Negative Sampling: 100%|██████████| 27485/27485 [00:58<00:00, 470.81it/s]
2026-01-07 18:34:01,197 - root - INFO - Saved negative samples to ./checkpoints/negative_test_size100_seed98765.pkl
2026-01-07 18:34:01,199 - root - INFO - Starting training...
2026-01-07 18:34:37,118 - root - INFO - Epoch 1/50
2026-01-07 18:34:37,119 - root - INFO -   Train Loss: 10.2512
2026-01-07 18:35:12,835 - root - INFO - Epoch 2/50
2026-01-07 18:35:12,836 - root - INFO -   Train Loss: 9.8501
2026-01-07 18:35:27,828 - root - INFO -   Validation Metrics:
2026-01-07 18:35:27,830 - root - INFO -     HitRate@5: 0.1661
2026-01-07 18:35:27,831 - root - INFO -     NDCG@5: 0.1088
2026-01-07 18:35:27,832 - root - INFO

Training completed!

Test Metrics:
  HitRate@5: 0.3094
  NDCG@5: 0.2219
  HitRate@10: 0.4157
  NDCG@10: 0.2560
  HitRate@15: 0.4924
  NDCG@15: 0.2763


In [ ]:
import torch

# Load the model from the .pth file
model_path = '/content/checkpoints/Bert4Rec.pth'  # Replace this with the actual path to your .pth file
model = torch.load(model_path)

# Print the model's contents
print(model)

# If the model is a dictionary (which it often is in PyTorch), you can print the keys:
if isinstance(model, dict):
    print(model.keys())


{'epoch': 23, 'model_state_dict': OrderedDict({'embedding.token.weight': tensor([[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.9063, -0.2860,  0.0074,  ..., -0.1073,  0.7072,  0.9347],
        [ 0.3107,  0.6826,  0.4041,  ...,  1.6442,  0.4073,  0.5238],
        ...,
        [ 0.5517,  0.4141, -0.0140,  ...,  0.6718,  0.8715,  0.8288],
        [ 1.3639,  0.2719,  0.2683,  ...,  2.5966,  0.2073, -0.0602],
        [-0.8387,  1.1410,  1.7294,  ...,  1.8730,  2.3333,  0.1084]],
       device='cuda:0'), 'embedding.position.pe.weight': tensor([[-0.5493,  1.2369, -1.6865,  ...,  0.0037, -0.6289, -0.4898],
        [ 1.9252, -1.0805, -0.7585,  ...,  0.0947,  0.6132, -0.1735],
        [-1.2936,  0.6227,  1.2160,  ..., -0.3457, -0.2602,  0.2031],
        ...,
        [ 0.1990, -0.8841, -0.8225,  ..., -1.1235,  0.9379,  0.8215],
        [ 0.1161, -0.7003, -0.4936,  ...,  1.1851,  0.0640, -2.0290],
        [ 0.1949, -0.1492,  0.2785,  ..., -0.3968,  0.7158,  0.6392]],
  